# Stage 1 wide noise classifiers (canonical)

SPL **50 / 60 / 70 / 80** only. **Train/Fit** → **Validate** row-level accuracy for HP / RF-vs-LR selection → **Youden threshold on Fit+Validate animals** (full calibration) → **Test** animal acc + AUC.

Summary table reports **fit / val / non-test / test** metrics. `non_test_acc_animal` is **animal-level only** (mean row proba per animal → Youden on Fit+Val → one label per `animal_id`, broadcast to wide/long rows as `noise_preds`).

Artifacts: `figures/cache/`.


In [1]:
import importlib
import sklearn

import utils.nn_stage2 as nn2
import utils.nn_stage2_data as nn2d
import utils.stage1_wide_report as s1r

importlib.reload(nn2d)
importlib.reload(nn2)
importlib.reload(s1r)

from utils.nn_stage2 import (
    fit_stage1_wide_best,
    fit_stage1_wide_logistic,
    fit_stage1_wide_rf,
)
from utils.nn_stage2_data import (
    STAGE_SPL_LEVELS,
    load_nn_stage2_data,
    splits_for_long_stage2,
    wide_columns_at_stage_spl,
    wide_stage1_fit,
    wide_stage1_val,
)
from utils.stage1_wide_report import export_stage1_artifacts, stage1_summary_table

In [2]:
data = load_nn_stage2_data()
sp = splits_for_long_stage2(data)
bb_tr = sp["bb_wide_train"]
bb_te = sp["bb_wide_test"]
lib_tr = sp["lib_train"]
lib_te = sp["lib_test"]

for name, cols in [("BB", bb_tr.columns), ("Lib", lib_tr.columns)]:
    spl = wide_columns_at_stage_spl(cols)
    print(
        f"{name}: {len(spl)} SPL 50-80 pivot cols (expect >0); levels={STAGE_SPL_LEVELS}"
    )

BB: 40 SPL 50-80 pivot cols (expect >0); levels=(50, 60, 70, 80)
Lib: 40 SPL 50-80 pivot cols (expect >0); levels=(50, 60, 70, 80)


In [3]:
bb_fit = wide_stage1_fit(bb_tr)
bb_val = wide_stage1_val(bb_tr)
lib_fit = wide_stage1_fit(lib_tr)
lib_val = wide_stage1_val(lib_tr)

print("Brad Buran wide (RF)")
s1_bb_rf = fit_stage1_wide_rf(
    bb_fit, bb_val, bb_te, data.noise_num_bb, data.noise_log_bb
)
print("\nBrad Buran wide (LR)")
s1_bb_lr = fit_stage1_wide_logistic(
    bb_fit, bb_val, bb_te, data.noise_num_bb, data.noise_log_bb
)
print("\nBrad Buran wide (best)")
s1_bb_best = fit_stage1_wide_best(
    bb_fit, bb_val, bb_te, data.noise_num_bb, data.noise_log_bb
)
display(stage1_summary_table(s1_bb_best["candidates"], s1_bb_best["stage1_model"]))

Brad Buran wide (RF)
           Noise RF  — fit acc (row): 0.817 | fit acc (animal@cal): 0.944 | val acc (row): 0.698 | val acc (animal@cal): 0.889 | non-test acc (animal@cal): 0.933 | non-test AUC (animal): 0.968 | threshold (Fit+Val Youden): 0.423 | threshold (Val-only Youden): 0.423 | test acc (animal@cal): 0.917 | test acc@0.5: 0.583 | AUC: 0.944

Brad Buran wide (LR)
           Noise LR  — fit acc (row): 0.700 | fit acc (animal@cal): 0.833 | val acc (row): 0.698 | val acc (animal@cal): 0.889 | non-test acc (animal@cal): 0.844 | non-test AUC (animal): 0.914 | threshold (Fit+Val Youden): 0.444 | threshold (Val-only Youden): 0.444 | test acc (animal@cal): 0.917 | test acc@0.5: 0.500 | AUC: 0.972

Brad Buran wide (best)
           Selected RF  — fit acc (animal@cal): 0.944 | non-test acc (animal@cal): 0.933 | threshold (Fit+Val): 0.423 | test acc (animal@cal): 0.917 | test acc@0.5: 0.583 | AUC: 0.944


,model,fit_acc_row,fit_acc_animal,val_acc_row,val_acc_animal,val_acc_animal_val_thr,non_test_acc_animal,non_test_auc_animal,threshold_fit_val_youden,threshold_val_only_youden,test_acc_animal,test_acc_animal_0p5,test_auc_animal,selected
0,RF,0.817391,0.944444,0.698113,0.888889,0.888889,0.933333,0.968,0.423256,0.423256,0.916667,0.583333,0.944444,True
1,LR,0.700000,0.833333,0.698113,0.888889,0.888889,0.844444,0.914,0.443867,0.443867,0.916667,0.500000,0.972222,False


In [4]:
print("Liberman wide (RF)")
s1_lib_rf = fit_stage1_wide_rf(
    lib_fit, lib_val, lib_te, data.noise_num_lib, data.noise_log_lib
)
print("\nLiberman wide (LR)")
s1_lib_lr = fit_stage1_wide_logistic(
    lib_fit, lib_val, lib_te, data.noise_num_lib, data.noise_log_lib
)
print("\nLiberman wide (best)")
s1_lib_best = fit_stage1_wide_best(
    lib_fit, lib_val, lib_te, data.noise_num_lib, data.noise_log_lib
)
display(stage1_summary_table(s1_lib_best["candidates"], s1_lib_best["stage1_model"]))

Liberman wide (RF)
           Noise RF  — fit acc (row): 0.927 | fit acc (animal@cal): 0.971 | val acc (row): 0.702 | val acc (animal@cal): 0.812 | non-test acc (animal@cal): 0.940 | non-test AUC (animal): 0.986 | threshold (Fit+Val Youden): 0.556 | threshold (Val-only Youden): 0.590 | test acc (animal@cal): 0.952 | test acc@0.5: 0.810 | AUC: 0.909

Liberman wide (LR)
           Noise LR  — fit acc (row): 0.668 | fit acc (animal@cal): 0.779 | val acc (row): 0.691 | val acc (animal@cal): 0.875 | non-test acc (animal@cal): 0.798 | non-test AUC (animal): 0.883 | threshold (Fit+Val Youden): 0.561 | threshold (Val-only Youden): 0.574 | test acc (animal@cal): 0.905 | test acc@0.5: 0.810 | AUC: 0.945

Liberman wide (best)
           Selected RF  — fit acc (animal@cal): 0.971 | non-test acc (animal@cal): 0.940 | threshold (Fit+Val): 0.556 | test acc (animal@cal): 0.952 | test acc@0.5: 0.810 | AUC: 0.909


,model,fit_acc_row,fit_acc_animal,val_acc_row,val_acc_animal,val_acc_animal_val_thr,non_test_acc_animal,non_test_auc_animal,threshold_fit_val_youden,threshold_val_only_youden,test_acc_animal,test_acc_animal_0p5,test_auc_animal,selected
0,RF,0.926952,0.970588,0.702128,0.8125,0.7500,0.940476,0.985795,0.555764,0.590417,0.952381,0.809524,0.909091,True
1,LR,0.667506,0.779412,0.691489,0.8750,0.8125,0.797619,0.882955,0.560730,0.574155,0.904762,0.809524,0.945455,False


In [5]:
paths = export_stage1_artifacts(
    s1_bb_best,
    s1_lib_best,
    s1_bb_rf,
    s1_bb_lr,
    s1_lib_rf,
    s1_lib_lr,
    sklearn_version=sklearn.__version__,
)
for k, p in paths.items():
    print(k, p.resolve())

metrics /Users/nowaki027/MSDS/Practicum/figures/cache/stage1_wide_metrics.json
shim /Users/nowaki027/MSDS/Practicum/figures/cache/stage1_wide_rf_metrics.json
eval_parquet /Users/nowaki027/MSDS/Practicum/figures/cache/stage1_wide_noise_lr_rf_eval.parquet
meta /Users/nowaki027/MSDS/Practicum/figures/cache/stage1_wide_noise_lr_rf_meta.json
